# pyppann: Neighbor-Preserving Approximate Nearest Neighbors

This notebook demonstrates pyppann's value proposition: **NeighborPreservingANN** achieves better recall than standard dimensionality reduction baselines by directly optimizing for neighbor preservation.

We compare 3 projection-based methods:
- **RandomProjection**: Gaussian random projections + brute kNN
- **PCA**: Principal component analysis + brute kNN  
- **NeighborPreservingANN**: Projection pursuit optimized for neighbor recall + brute kNN

In [ ]:
import time

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.neighbors import NearestNeighbors

from pyppann import (
    NeighborPreservingANN,
    PCAAnn,
    RandomProjectionANN,
    compute_recall,
)

In [ ]:
np.random.seed(42)

n_samples = 2000
n_features = 50
n_queries = 200
k = 10
n_components = 20

X, _ = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=30,
    n_redundant=20,
    n_clusters_per_class=3,
    random_state=42,
)
X = X.astype(np.float64)

query_indices = np.random.choice(n_samples, n_queries, replace=False)
X_query = X[query_indices]

print(f"Data: {n_samples} samples, {n_features} features")
print(f"Queries: {n_queries} points")
print(f"k={k} neighbors, {n_components} projection components")

In [ ]:
nn_exact = NearestNeighbors(n_neighbors=k + 1, algorithm="brute")
nn_exact.fit(X)
_, true_neighbors_full = nn_exact.kneighbors(X_query)
true_neighbors = true_neighbors_full[:, 1:]

methods = {
    "RandomProjection": RandomProjectionANN(n_components=n_components, random_state=42),
    "PCA": PCAAnn(n_components=n_components, random_state=42),
    "NeighborPreservingANN": NeighborPreservingANN(
        n_components=n_components,
        k=k,
        n_candidates=50,
        n_refinement_steps=30,
        n_backfit_iters=2,
        subsample_size=min(1000, n_samples),
        use_pca_init=True,
        random_state=42,
        verbose=False,
    ),
}

results = []

for name, model in methods.items():
    print(f"Evaluating {name}...")
    
    fit_start = time.perf_counter()
    model.fit(X, k=k)
    fit_time = time.perf_counter() - fit_start
    
    query_start = time.perf_counter()
    pred_neighbors = np.asarray(model.kneighbors(X_query, n_neighbors=k, return_distance=False))
    query_time = time.perf_counter() - query_start
    
    recall = compute_recall(true_neighbors, pred_neighbors)
    
    results.append({
        "Method": name,
        "Recall@k": recall,
        "Fit Time (s)": fit_time,
        "Query Time (ms)": query_time * 1000,
    })
    print(f"  Recall@{k}: {recall:.4f}")

In [ ]:
df = pd.DataFrame(results)
df = df.sort_values("Recall@k", ascending=False)
df["Recall@k"] = df["Recall@k"].map("{:.4f}".format)
df["Fit Time (s)"] = df["Fit Time (s)"].map("{:.2f}".format)
df["Query Time (ms)"] = df["Query Time (ms)"].map("{:.2f}".format)
df = df.reset_index(drop=True)
df

## Interpretation

**Key takeaway**: NeighborPreservingANN achieves higher recall than PCA and RandomProjection baselines because it directly optimizes for neighbor preservation during projection learning.

- **RandomProjection** uses random Gaussian directions that don't account for data structure
- **PCA** captures maximum variance but not necessarily neighborhood structure
- **NeighborPreservingANN** learns projections that specifically preserve k-nearest neighbor relationships

The trade-off: NeighborPreservingANN has a longer fit time due to its optimization process, but achieves better recall for the same number of dimensions.